# 07 · XGBoost with Random Rotation Features (Lending Club)

Tests whether applying a random orthogonal rotation to the numerical feature space affects XGBoost's performance. Because XGBoost makes axis-aligned splits, an orthogonal rotation in principle changes what each split can express; if the rotated model performs comparably, the numerical features already lie close to split-friendly directions.

Both an FTT comparison (which is largely permutation-invariant in the numerical embedding) and an XGB comparison are included.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/processed_data_densest.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/lending_club")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Lending Club dataset.
df4 = pd.read_parquet(DATA_PATH)

train_val_df, test_df = train_test_split(
    df4, test_size=0.2,
    stratify=df4["target_binary"], random_state=RANDOM_SEED,
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    stratify=train_val_df["target_binary"], random_state=RANDOM_SEED,
)
print("Train_val / test sizes:", len(train_val_df), len(test_df))
print("Train / val sizes:", len(train_df), len(val_df))

cat_columns = [
    "term", "emp_length", "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state", "application_type",
    "initial_list_status", "disbursement_method",
]
num_cols = [
    c for c in list(df4.columns)
    if c not in cat_columns + ["id", "emp_title", "target_binary"]
]
print(f"# numerical features: {len(num_cols)}  |  # categorical: {len(cat_columns)}")

## 3. Preprocessing & Rotation

Median impute + standard-scale numerical features, then build both unrotated and rotated XGB-friendly frames.

In [ ]:
# Build per-split numerical frames with median imputation + standardization.
num_impute = {}
scaler = StandardScaler()
train_num = train_df[num_cols].copy()
for c in train_num.columns:
    if train_num[c].dtype == "object":
        train_num[c] = pd.to_numeric(train_num[c], errors="coerce")
    med = train_num[c].median()
    train_num[c] = train_num[c].fillna(med)
    num_impute[c] = med
train_num[train_num.columns] = scaler.fit_transform(train_num[train_num.columns])

def _impute_scale(df_):
    df_part = df_[num_cols].copy()
    for c in df_part.columns:
        if df_part[c].dtype == "object":
            df_part[c] = pd.to_numeric(df_part[c], errors="coerce")
        df_part[c] = df_part[c].fillna(num_impute[c])
    df_part[df_part.columns] = scaler.transform(df_part[df_part.columns])
    return df_part

val_num  = _impute_scale(val_df)
test_num = _impute_scale(test_df)

# Build XGB-friendly frames (raw scaled numeric + categorical-as-category).
train_df_xgb = pd.concat(
    [train_num.reset_index(drop=True),
     train_df[cat_columns].reset_index(drop=True).astype("category")],
    axis=1,
)
train_df_xgb["target"] = train_df["target_binary"].values
val_df_xgb = pd.concat(
    [val_num.reset_index(drop=True),
     val_df[cat_columns].reset_index(drop=True).astype("category"),
     val_df["target_binary"].reset_index(drop=True)],
    axis=1,
)
test_df_xgb = pd.concat(
    [test_num.reset_index(drop=True),
     test_df[cat_columns].reset_index(drop=True).astype("category"),
     test_df["target_binary"].reset_index(drop=True)],
    axis=1,
)
print("XGB frames ready.")

In [ ]:
# Random orthogonal rotation of the numerical feature space (variance-preserving).
rng = np.random.default_rng(RANDOM_SEED)
n_num = len(num_cols)
A = rng.standard_normal((n_num, n_num))
Q, _ = np.linalg.qr(A)  # Q is orthonormal -> a rotation matrix.

def _rotate(num_arr):
    return num_arr @ Q

train_df_rot = pd.DataFrame(
    _rotate(train_num.to_numpy()),
    columns=[f"rot_{i}" for i in range(n_num)],
    index=train_num.index,
)
val_df_rot = pd.DataFrame(
    _rotate(val_num.to_numpy()),
    columns=[f"rot_{i}" for i in range(n_num)],
    index=val_num.index,
)
test_df_rot = pd.DataFrame(
    _rotate(test_num.to_numpy()),
    columns=[f"rot_{i}" for i in range(n_num)],
    index=test_num.index,
)

train_df_xgb_rot = pd.concat(
    [train_df_rot.reset_index(drop=True),
     train_df[cat_columns].reset_index(drop=True).astype("category")],
    axis=1,
)
train_df_xgb_rot["target"] = train_df["target_binary"].values
val_df_xgb_rot = pd.concat(
    [val_df_rot.reset_index(drop=True),
     val_df[cat_columns].reset_index(drop=True).astype("category"),
     val_df["target_binary"].reset_index(drop=True)],
    axis=1,
)
test_df_xgb_rot = pd.concat(
    [test_df_rot.reset_index(drop=True),
     test_df[cat_columns].reset_index(drop=True).astype("category"),
     test_df["target_binary"].reset_index(drop=True)],
    axis=1,
)
print("Rotation-augmented frames ready.")

## 4. XGBoost — Baseline (Unrotated)

In [ ]:
EARLY_STOPPING_ROUNDS = 50
best_params = {'n_estimators': 2496,
 'learning_rate': 0.049201866528934435,
 'max_depth': 3,
 'lambda': 1.5845691933705806,
 'alpha': 0.25801123987508096,
 'scale_pos_weight': 1.995383392364703,
 'min_child_weight': 21,
 'gamma': 0.013272907579881622,
 'subsample': 0.6000572006728725,
 'colsample_bytree': 0.727384622457453}
final_xgb_config = {
    **best_params, # Unpack the parameters found by Optuna
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "enable_categorical" : True,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "device":"cuda",
}

In [ ]:
import xgboost as xgb
model_xgb = xgb.XGBClassifier(**final_xgb_config)

model_xgb.fit(
    train_df_xgb.drop(columns=['target']), train_df_xgb['target'],
    eval_set=[(val_df_xgb.drop(columns=['target_binary']), val_df_xgb['target_binary'])],
    verbose=50)

val_pred = model_xgb.predict_proba(val_df_xgb.drop(columns=['target_binary']))[:, 1]
roc_auc_score(val_df_xgb['target_binary'], val_pred)

In [ ]:
val_pred = model_xgb.predict_proba(val_df_xgb.drop(columns=['target_binary']))[:, 1]
roc_auc_score(val_df_xgb['target_binary'], val_pred)

## 5. XGBoost — Rotated Features

In [ ]:
import xgboost as xgb
model_rot = xgb.XGBClassifier(**final_xgb_config)
model_rot.fit(
    train_df_xgb_rot.drop(columns=['target']), train_df_xgb_rot['target'],
    eval_set=[(val_df_xgb_rot.drop(columns=['target_binary']), val_df_xgb_rot['target_binary'])],
    verbose=False,
)

val_pred = model_rot.predict_proba(val_df_xgb_rot.drop(columns=['target_binary']))[:, 1]
roc_auc_score(val_df_xgb_rot['target_binary'], val_pred)

## 6. FTT on Rotated Features

Sanity-check: train an FT-Transformer on the rotated numerical features. FTT's numerical embedding is a per-feature linear layer, so rotation should be (nearly) absorbed by the embedding weights after a few epochs.

In [ ]:
train_df_rot_ftt = pd.concat([train_df_rot.reset_index(drop=True), train_df[cat_columns].reset_index(drop=True)], axis=1)
num_impute = {}
scaler = StandardScaler()
if num_cols:
    num_df = train_df_rot_ftt[num_cols].copy()
    for c in num_df.columns:
        if num_df[c].dtype == 'object':
            num_df[c] = pd.to_numeric(num_df[c], errors='coerce')
        med = num_df[c].median()
        num_df[c] = num_df[c].fillna(med)
        num_impute[c] = med
    num_df[num_df.columns] = scaler.fit_transform(num_df[num_df.columns])
else:
    num_df = pd.DataFrame(index=train_df_rot_ftt.index)
cat_encoders: Dict[str, LabelEncoder] = {}
cat_cardinalities = []
cat_df = train_df_rot_ftt[cat_columns].copy()
for c in cat_columns:
    le = LabelEncoder()
    series = cat_df[c].fillna("MISSING").astype(str)
    le.fit(series)
    new_classes = list(le.classes_)
    if "MISSING" not in new_classes:
        new_classes.append("MISSING")
    new_classes.append("UNKNOWN")
    le.classes_ = np.array(new_classes)
    cat_df[c + "_le"] = le.transform(series)
    cat_encoders[c] = le
    cat_cardinalities.append(len(le.classes_))
# cat_df

train_df_proc = pd.concat([num_df.reset_index(drop=True), cat_df[[c + "_le" for c in cat_columns]].reset_index(drop=True),train_df["target_binary"].reset_index(drop=True)], axis=1)
# train_df_proc["target"] =
# train_df_proc
print('encoded train')

In [ ]:
val_df_rot_ftt = pd.concat([val_df_rot.reset_index(drop=True), val_df[cat_columns].reset_index(drop=True), val_df['target_binary'].reset_index(drop=True)], axis=1)
# val_df = val_df[num_cols+[c + "_le" for c in cat_columns]+['target_binary']]

test_df_rot_ftt = pd.concat([test_df_rot.reset_index(drop=True), test_df[cat_columns].reset_index(drop=True), test_df['target_binary'].reset_index(drop=True)], axis=1)

val_num = val_df_rot_ftt[num_cols].copy()
for c in val_num.columns:
    if val_num[c].dtype == 'object':
        val_num[c] = pd.to_numeric(val_num[c], errors='coerce')

for c in val_num.columns:
    val_num[c] = val_num[c].fillna(num_impute[c])

val_num = scaler.transform(val_num)
val_df_rot_ftt[num_cols] = val_num

for c in cat_columns:
    le = cat_encoders[c]
    classes = le.classes_
    mapping = {cls: i for i, cls in enumerate(classes)}
    unknown_id = mapping["UNKNOWN"]
    series = val_df_rot_ftt[c].fillna("MISSING").astype(str)
    val_df_rot_ftt[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

val_df = val_df_rot_ftt[num_cols+[c + "_le" for c in cat_columns]+['target_binary']]
print('encoded val')

test_num = test_df_rot_ftt[num_cols].copy()
for c in test_num.columns:
    if test_num[c].dtype == 'object':
        test_num[c] = pd.to_numeric(test_num[c], errors='coerce')

for c in test_num.columns:
    test_num[c] = test_num[c].fillna(num_impute[c])

test_num = scaler.transform(test_num)
test_df_rot_ftt[num_cols] = test_num

for c in cat_columns:
    le = cat_encoders[c]
    classes = le.classes_
    mapping = {cls: i for i, cls in enumerate(classes)}
    unknown_id = mapping["UNKNOWN"]
    series = test_df_rot_ftt[c].fillna("MISSING").astype(str)
    test_df_rot_ftt[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

print('encoded test')

test_df = test_df_rot_ftt[num_cols+[c + "_le" for c in cat_columns]+['target_binary']]
cat_le_cols = [c + "_le" for c in cat_columns]
d_numerical = len(num_cols)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_numpy = { "train": {},'val':{},'test':{}}
data_numpy["train"]["x_cat"] = train_df_proc[cat_le_cols]
data_numpy["val"]["x_cat"] = val_df[cat_le_cols]
data_numpy["test"]["x_cat"] = test_df[cat_le_cols]

data_numpy["train"]["X_cont"] = train_df_proc[num_cols]
data_numpy["val"]["X_cont"] = val_df[num_cols]
data_numpy["test"]["X_cont"] = test_df[num_cols]

data_numpy["train"]["y"] = train_df_proc['target_binary']
data_numpy["val"]["y"] = val_df['target_binary']
data_numpy["test"]["y"] = test_df['target_binary']

data = {
    part: {k: torch.as_tensor(v.to_numpy()).to(device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}
d_numerical = len(num_cols)

In [ ]:
data={}
for part in data_numpy:
    data[part] = {}

    # numerical features
    data[part]["X_cont"] = torch.tensor(
        data_numpy[part]["X_cont"].to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    # categorical features
    data[part]["x_cat"] = torch.tensor(
        data_numpy[part]["x_cat"].to_numpy(),
        dtype=torch.long,
        device=device,
    )

    # labels
    data[part]["y"] = torch.tensor(
        data_numpy[part]["y"].to_numpy(),
        dtype=torch.float32,
        device=device,
    )

In [ ]:
BATCH_SIZE=2048+4096
trial_results=[]
global_best_auc=0.73
import math
from rtdl_revisiting_models import FTTransformer

# global_model_path = str(MODELS_DIR / "ftt_rotation_lc.pth")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities,
    n_blocks=5, #4
    d_block=64, #192
    attention_n_heads=4,#8
    ffn_d_hidden_multiplier=8/3,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)
n_epochs = 2 if DEV_MODE else 100
patience = 10
Lr = 0.0002
optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.3,
anneal_strategy="cos"
)

loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return auc # The higher -- the better.


print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


batch_size = BATCH_SIZE
epoch_size = math.ceil(train_df.shape[0] / batch_size)
timer = delu.tools.Timer()
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            torch.tensor(3.5, device=y_batch.device),
            torch.tensor(1.0, device=y_batch.device)
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    trial_results.append({
    "epoch_n": epoch,
    "auc_val": val_auc,
    "auc_test": test_auc,
    'best':best,
    'time':timer.__str__(),
    'auc_train':train_auc
      })
    # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch, 'train':train_auc}
        # if val_auc>global_best_auc:
        # torch.save(model.state_dict(), global_model_path)
        # global_best_auc=val_auc

    print()

In [ ]:
torch.save(model.state_dict(),str(MODELS_DIR / 'ftt_rotation_lc.pth'))